# 02 — Baselines: regressão logística (features) e SimpleCNN

Validação cruzada **por paciente** (4 folds) no conjunto de desenvolvimento (todos os centros menos o 2).
Reportamos **média ± desvio entre os folds**. O centro 2 (teste externo) só é usado **uma vez**, no final.

Se os resultados já existirem em `outputs/`, eles são carregados; senão o treino roda aqui
(o mesmo que `python pathoshift.py train --model ...`).

In [ ]:
import sys
from pathlib import Path

# Raiz do repositório (funciona rodando de notebooks/ ou da raiz, local ou no Colab)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import utils
from src.data import load_hf_dataset, read_image
from src.manifest import load_manifest, summarize

cfg = utils.load_config(ROOT / "configs" / "default.yaml")
utils.set_seed(cfg["seed"])
man = load_manifest(utils.resolve_path(cfg["paths"]["manifest"]))
ds = load_hf_dataset(cfg)

from src import train
from src.evaluate import compute_metrics, mean_std_text, print_metrics
from src.viz import error_grid, plot_confusion_matrix

OUT = utils.resolve_path(cfg["paths"]["outputs_dir"])


def cv_results(model_name):
    """Carrega a validação cruzada salva em outputs/<modelo>/ (ou roda, se não existir)."""
    if not (OUT / model_name / "cv_oof.csv").exists():
        train.run_cv(cfg, model_name)
    table = pd.read_csv(OUT / model_name / "cv_folds.csv", index_col=0)
    oof = pd.read_csv(OUT / model_name / "cv_oof.csv")
    return table, oof

## 1. Baseline clássico: StandardScaler + LogisticRegression(class_weight='balanced')

In [ ]:
tab_lr, oof_lr = cv_results("logreg")
display(tab_lr.round(3))
print(mean_std_text(tab_lr).to_string())

In [ ]:
m = compute_metrics(oof_lr["label"], oof_lr["prob"])
print_metrics(m, "logreg — predições de validação dos 4 folds juntas")
plot_confusion_matrix(m["confusion_matrix"], "logreg (CV)");

### Falsos positivos (normal → tumor) e falsos negativos (tumor → normal) mais confiantes

In [ ]:
error_grid(ds, oof_lr, "fp", n=12);
error_grid(ds, oof_lr, "fn", n=12);

## 2. SimpleCNN (Conv → Pooling → Dense, a mesma rede do slide)

In [ ]:
tab_cnn, oof_cnn = cv_results("cnn")
display(tab_cnn.round(3))
print(mean_std_text(tab_cnn).to_string())

In [ ]:
hist = pd.read_csv(OUT / "cnn" / "cv_history.csv")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for k, h in hist.groupby("fold"):
    axes[0].plot(h["epoch"], h["train_loss"], marker="o", label=f"fold {k}")
    axes[1].plot(h["epoch"], h["val_f1_macro"], marker="o")
    axes[2].plot(h["epoch"], h["tempo_s"], marker="o")
for ax, t in zip(axes, ["loss de treino", "F1 macro (validação)", "tempo por época (s)"]):
    ax.set_title(t)
    ax.set_xlabel("época")
axes[0].legend();

In [ ]:
m = compute_metrics(oof_cnn["label"], oof_cnn["prob"])
print_metrics(m, "CNN — predições de validação dos 4 folds juntas")
plot_confusion_matrix(m["confusion_matrix"], "CNN (CV)");
error_grid(ds, oof_cnn, "fp", n=12);
error_grid(ds, oof_cnn, "fn", n=12);

## 3. Comparação (média ± desvio entre os folds)

In [ ]:
pd.DataFrame({"logreg": mean_std_text(tab_lr), "SimpleCNN": mean_std_text(tab_cnn)})

## 4. Por que o desvio é tão grande? Recall de tumor por paciente
O dev tem poucos pacientes com tumor, então cada fold depende de "qual paciente caiu nele".

In [ ]:
def recall_por_paciente(oof):
    t = oof[oof["label"] == 1]
    return t.groupby(["fold", "centre_id", "patient_id", "node_id"])["prob"].agg(
        n="size", prob_media="mean", recall=lambda p: (p >= 0.5).mean())

pd.concat({"logreg": recall_por_paciente(oof_lr), "cnn": recall_por_paciente(oof_cnn)}, axis=1).round(3)

## 5. Teste externo (centro 2) — rodar UMA vez, no final do trabalho
Treina no dev inteiro e avalia no centro 2. Só mude a flag quando os modelos estiverem decididos:
olhar o teste e voltar a ajustar o modelo invalida o teste.

In [ ]:
RODAR_TESTE_FINAL = False

if RODAR_TESTE_FINAL:
    for nome in ["logreg", "cnn"]:
        m = train.run_final(cfg, nome)
        plot_confusion_matrix(m["confusion_matrix"], f"{nome} — centro {cfg['splits']['test_centre']}")
else:
    print("Teste externo não executado (RODAR_TESTE_FINAL = False).")